# DiaParser
**Direct Attentive Dependency Parser**

In [ ]:
!pip install diaparser

In [35]:
from diaparser.parsers import Parser

### Create a parser for Italian

Load a pretrained model for Italian, named `it_isdt.dbmdz-xxl`, i.e. a parser trained on the Italian ISDT treebank, using the transformner model `dbmdz/bert-base-italian-xxl-cased`.

The model will be downloaded and cached locally for further use.

In [2]:
parser = Parser.load('it_isdt.dbmdz-xxl')

Using bos_token, but it is not set yet.
Using eos_token, but it is not set yet.


Alternatively, we can just specify the language and accept a default model:

In [34]:
parser = Parser.load('', lang='it')

Using bos_token, but it is not set yet.
Using eos_token, but it is not set yet.


You may parse plain text, by telling the language used: 

In [3]:
dataset = parser.predict('Lo schermo è buono, ma la batteria dura poco.', text='it')

`dataset` is an instance of `diaparser.utils.Dataset` containing the parse trees for each sentence.

Let's look at the first one:

In [4]:
dataset.sentences[0]

# sent_id = 1
# text = Lo schermo è buono, ma la batteria dura poco.
1	Lo	_	_	_	_	2	det	_	_
2	schermo	_	_	_	_	4	nsubj	_	_
3	è	_	_	_	_	4	cop	_	_
4	buono	_	_	_	_	0	root	_	_
5	,	_	_	_	_	9	punct	_	_
6	ma	_	_	_	_	9	cc	_	_
7	la	_	_	_	_	8	det	_	_
8	batteria	_	_	_	_	9	nsubj	_	_
9	dura	_	_	_	_	4	conj	_	_
10	poco	_	_	_	_	9	advmod	_	_
11	.	_	_	_	_	4	punct	_	_

## Display parse tree

In [5]:
from spacy import displacy

def display(sent):
    displacy.render(sent.to_json(), style='dep', manual=True,
                    options={'compact': True, 'distance': 120, 
                             'word_spacing': 20, 'offset_x':20})

In [6]:
display(dataset.sentences[0])

## English
Load a pretrained model for English, named `en_ewt.electra-base`, i.e. a parser trained on the English EWT treebank, using the transformner model `electra-base-disciminator`.

In [7]:
parser_en = Parser.load('en_ewt.electra-base')

Using bos_token, but it is not set yet.
Using eos_token, but it is not set yet.


You may parse plain text, by telling the language used: 

In [8]:
dataset = parser_en.predict('I like the display, but the battery life is short.', text='en')

`dataset` is an instance of `diaparser.utils.Dataset` containing the predicted syntactic trees.

Let's look at the first one:

In [9]:
dataset.sentences[0]

# sent_id = 1
# text = I like the display, but the battery life is short.
1	I	_	_	_	_	2	nsubj	_	_
2	like	_	_	_	_	0	root	_	_
3	the	_	_	_	_	4	det	_	_
4	display	_	_	_	_	2	obj	_	_
5	,	_	_	_	_	11	punct	_	_
6	but	_	_	_	_	11	cc	_	_
7	the	_	_	_	_	9	det	_	_
8	battery	_	_	_	_	9	compound	_	_
9	life	_	_	_	_	11	nsubj	_	_
10	is	_	_	_	_	11	cop	_	_
11	short	_	_	_	_	2	conj	_	_
12	.	_	_	_	_	2	punct	_	_

In [10]:
display(dataset.sentences[0])

## Parse from tokenized text

Or you can provide tokenized text, as weel ask to see the estimated probabiity for each predicted arc:

In [11]:
dataset = parser_en.predict(['I', 'liked', 'the', 'display', '.'], prob=True)

You may then look at individual fields of the tokens in a sentence and the probability of their arcs.

In [12]:
import torch
print(f"arcs:  {dataset.arcs[0]}\n"
      f"rels:  {dataset.rels[0]}\n"
      f"probs: {dataset.probs[0].gather(1,torch.tensor(dataset.arcs[0]).unsqueeze(1)).squeeze(-1)}")

arcs:  [2, 0, 4, 2, 2]
rels:  ['nsubj', 'root', 'det', 'obj', 'punct']
probs: tensor([1.0000, 1.0000, 1.0000, 1.0000, 0.9999])


# Information Extraction

From tha AODA act of Ontario: https://www.aoda.ca/integrated/#etrame

In [13]:
art16 = """In addition to the requirements under section 7, obligated organizations that are school boards
or educational or training institutions shall provide educators with accessibility awareness training
related to accessible program or course delivery and instruction.
Obligated organizations that are school boards or educational or training institutions shall keep
a record of the training provided under this section, including the dates on which the training
is provided and the number of individuals to whom it is provided."""

In [14]:
output = parser_en.predict(art16, text='en')

In [15]:
output.sentences

[# sent_id = 1
 # text = In addition to the requirements under section 7, obligated organizations that are school boards or educational or training institutions shall provide educators with accessibility awareness training related to accessible program or course delivery and instruction.
 1	In	_	_	_	_	2	case	_	_
 2	addition	_	_	_	_	22	obl	_	_
 3	to	_	_	_	_	5	case	_	_
 4	the	_	_	_	_	5	det	_	_
 5	requirements	_	_	_	_	2	nmod	_	_
 6	under	_	_	_	_	7	case	_	_
 7	section	_	_	_	_	5	nmod	_	_
 8	7	_	_	_	_	7	nummod	_	_
 9	,	_	_	_	_	22	punct	_	_
 10	obligated	_	_	_	_	11	amod	_	_
 11	organizations	_	_	_	_	22	nsubj	_	_
 12	that	_	_	_	_	15	nsubj	_	_
 13	are	_	_	_	_	15	cop	_	_
 14	school	_	_	_	_	15	compound	_	_
 15	boards	_	_	_	_	11	acl:relcl	_	_
 16	or	_	_	_	_	20	cc	_	_
 17	educational	_	_	_	_	20	amod	_	_
 18	or	_	_	_	_	19	cc	_	_
 19	training	_	_	_	_	17	conj	_	_
 20	institutions	_	_	_	_	15	conj	_	_
 21	shall	_	_	_	_	22	aux	_	_
 22	provide	_	_	_	_	0	root	_	_
 23	educators	_	_	_	_	22	obj	_	_
 24	with	_

In [16]:
display(output.sentences[0])

# Match a pattern

In [22]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("en_core_web_sm")

In [32]:
def visualise_subtrees(doc, subtrees):

    words = [{"text": t.text, "tag": t.pos_} for t in doc]

    if not isinstance(subtrees[0], list):
        subtrees = [subtrees]

    for subtree in subtrees:
        arcs = []
        tree_indices = set(subtree)
        for index in subtree:
            token = doc[index]
            head = token.head
            if token.head.i == token.i or token.head.i not in tree_indices:
                continue
            else:
                if token.i < head.i:
                    arcs.append(
                        {
                            "start": token.i,
                            "end": head.i,
                            "label": token.dep_,
                            "dir": "left",
                        }
                    )
                else:
                    arcs.append(
                        {
                            "start": head.i,
                            "end": token.i,
                            "label": token.dep_,
                            "dir": "right",
                        }
                    )
        print("Subtree: ", subtree)
        displacy.render(
            {"words": words, "arcs": arcs},
            style="dep",
            options={'compact': True, "distance": 120},
            manual=True
        )

### Pattern for a subtree

- rooted at a VERB (`keep`)
- with a MODAL auxiliary (`shall`)
- with a SUBJECT (`nsubj`)
- and an OBJECT (`dobj`)

In [17]:
pattern = [{'SPEC': {'NODE_NAME': 'root'},
            'PATTERN': {'POS': 'VERB'}},
           {'SPEC': {'NODE_NAME': 'MODAL', 'NBOR_RELOP': '>', 'NBOR_NAME': 'root'},
             'PATTERN': {'ORTH': 'shall'}},
           {'SPEC': {'NODE_NAME': 'SUBJ', 'NBOR_RELOP': '>', 'NBOR_NAME': 'root'},
             'PATTERN': {'DEP': 'nsubj'}},
           {'SPEC': {'NODE_NAME': 'OBJ', 'NBOR_RELOP': '>', 'NBOR_NAME': 'root'},
            'PATTERN': {'DEP': 'dobj'}}]

In [23]:
matcher = DependencyMatcher(nlp.vocab)
matcher.add("pattern", None, pattern)

Apply to second sentence from Art. 16

In [27]:
art16_2 = nlp("""Obligated organizations that are school boards or educational or training institutions shall keep a record of the training provided under this section, including the dates on which the training
is provided and the number of individuals to whom it is provided.""")

Show the parse tree

In [30]:
displacy.render(art16_2, options={'compact': True, 'distance': 120, 
                             'word_spacing': 20, 'offset_x':20})

Extract and visualize the subtree

In [33]:
match = matcher(art16_2)[0]
subtree = match[1][0]
visualise_subtrees(art16_2, subtree)

Subtree:  [12, 11, 1, 14]
